# Home Exercise on Linear Regression - Part 2
## Task Description
Implement a **logistic regression model** using *gradient descent, Adam, any other optimization method* to optimize the parameters or closed-form solution. (You can use other variations of Linear Regression, such as **Lasso, Ridge, etc**, but you must implement them yourself.)

 
* You are only allowed to use computational libraries such as NumPy, Math, etc for implementing the model and training process.
  
* You must not use machine learning libraries or frameworks like scikit-learn, TensorFlow, PyTorch, etc. that provide pre-built models.

* For other tasks (e.g., data processing, visualization), you are free to use any library.

After implementing the model, use it to solve the following problem: [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic/)

## Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, shutil

import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

## Download and load dataset into DataFrame

In [2]:
def moving_folder(path, des):
    if not os.path.exists(path) or not os.path.exists(des):
        print(f"Error: Source folder '{path}' not found.")
        return
    try:
        shutil.move(path, des)
        print(f"Successfully moved folder '{os.path.basename(path)}' to '{des}'")
    except shutil.Error as e:
        print(f"There are errors. Try again!")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# moving_folder("titanic.zip", "./data")

In [5]:
!kaggle competitions download -c titanic

Traceback (most recent call last):
  File "/home/dikhang/miniconda3/envs/llms/bin/kaggle", line 3, in <module>
    from kaggle.cli import main
  File "/home/dikhang/miniconda3/envs/llms/lib/python3.11/site-packages/kaggle/__init__.py", line 6, in <module>
    api.authenticate()
  File "/home/dikhang/miniconda3/envs/llms/lib/python3.11/site-packages/kaggle/api/kaggle_api_extended.py", line 434, in authenticate
    raise IOError('Could not find {}. Make sure it\'s located in'
OSError: Could not find kaggle.json. Make sure it's located in /home/dikhang/.config/kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/


In [3]:
import zipfile
def unzip_file(path, des, delete=True):
    if not os.path.exists(path) or not os.path.exists(des):
        print(f"Error: Source folder '{path}' not found.")
        return
    with zipfile.ZipFile(path) as zipref:
        zipref.extractall(des)
        print(f"Extracted zipfile in {des}")
    if delete:
        os.remove(path)
        return

In [ ]:
data_dir = "./data"
zip_path = os.path.join(data_dir, "titanic.zip")

unzip_file(zip_path, data_dir, True)

train_path = os.path.join(data_dir, "train.csv")
test_path = os.path.join(data_dir, "test.csv")

if os.path.exists(train_path) and os.path.exists(test_path):
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    print(f"Load dataset in to train and test DataFrame successfully. Train_df info is:\n{df_train.info()}")

else:
    print("There are errors.")

Extracted zipfile in ./data
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
Load dataset in to train and test DataFrame successfully. Train_df info is:
None


## Data preprocessing

### Checking raw dataset

In [8]:
print(df_train.head(5))
print("="*100)
print(df_train.info())
print("="*100)

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

In [9]:
print(df_train.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
